In [22]:
#/archive dataset from Kaggle: https://www.kaggle.com/datasets/bikashkundu/can-hcrl-otids
#/road dataset: https://zenodo.org/records/10462796

In [12]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split



In [13]:
#     DoS Attack (656,579) : Injecting messages of ‘0x000’ CAN ID in a short cycle.
#     Fuzzy Attack (591,990): Injecting messages of spoofed random CAN ID and DATA values.
#     Impersonation Attack (995,472): Injecting messages of Impersonating node, arbitration ID = '0x164'.
#     Attack Free State (2,369,868 ): Normal CAN messages.
   
# 'target' column doesn't specify the attacks, compute and label the values by seeing the above mentioned points 
# for each attacks which are there before applying any model. Impersonation_attacks can be said as spoofing attacks.

#balanced_mix = pd.read_csv("archive/CAN_HCRL_OTIDS_B.csv") #CAN_HCRL_OTIDS_B is combined records of all four files and oversampled with SMOTE.(Balanced)
unbalanced_mix = pd.read_csv("archive/CAN_HCRL_OTIDS_UB.csv") #CAN_HCRL_OTIDS_UB is combined records of all four files.(Un-Balanced)
# attack_free = pd.read_csv("archive/dataset.csv") #normal, target column is 0
# dos_attacks = pd.read_csv("archive/dataset1.csv") #dos attack,  target column as 1
# fuzzy_attacks = pd.read_csv("archive/dataset2.csv") #fuzzy attack target column as 2
# impersonation_attacks = pd.read_csv("archive/dataset3.csv") #impersonation attacks, target column as 3. 

In [15]:
unbalanced_mix = unbalanced_mix.fillna(0)
unbalanced_mix["ID1_int"] = unbalanced_mix["ID1"].apply(lambda x: int(str(x), 16))
encoder = LabelEncoder()
unbalanced_mix["ID1"] = encoder.fit_transform(unbalanced_mix["ID1"].astype(str)) # Encode the ID 
unbalanced_mix.loc[unbalanced_mix['target'] != 0, 'target'] = 1

In [16]:
X_raw = unbalanced_mix.drop(columns=["target"])
y = unbalanced_mix["target"].values
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)
X = np.array(X_scaled) #X = np.array(X_scaled).reshape(-1, X_scaled.shape[1], 1)


In [17]:
def create_windows_otids(X, y, window_sec=3, stride_sec=0.5, target_hz=200, label_policy="majority"):
    win_size = int(window_sec * target_hz)
    stride   = int(stride_sec * target_hz)
    seqs, labs = [], []

    for start in range(0, len(X) - win_size, stride):
        end = start + win_size
        window = X[start:end]
        labels = y[start:end]
        if label_policy == "last":
            lab = int(labels[-1])
        elif label_policy == "majority":
            lab = int(labels.mean() >= 0.5)
        elif label_policy == "any":
            lab = int(labels.sum() > 0)
        seqs.append(window)
        labs.append(lab)
    return np.array(seqs), np.array(labs)

# Example usage
X_seq, y_seq = create_windows_otids(X, y, window_sec=3, stride_sec=0.5, target_hz=200, label_policy='majority')
print(X_seq.shape)  # (num_windows, 600, num_features)
print(y_seq.shape)  # (num_windows,)

(42516, 600, 11)
(42516,)


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=44, shuffle=True, stratify=y_seq
)

In [ ]:
def standardize(X_train, y_train, X_test, y_test):
    # Step 1: Clip outliers (important for ROAD)
    X_train = np.clip(X_train, -1e6, 1e6)
    X_test  = np.clip(X_test,  -1e6, 1e6)
    # Step 2: Standardize features
    scaler = StandardScaler()
    X_train_flat = X_train.reshape(-1, X_train.shape[-1])
    X_test_flat  = X_test.reshape(-1,  X_test.shape[-1])
    X_train_scaled = scaler.fit_transform(X_train_flat)
    X_test_scaled  = scaler.transform(X_test_flat)
    X_train = X_train_scaled.reshape(X_train.shape)
    X_test  = X_test_scaled.reshape(X_test.shape)
    return X_train, y_train, X_test, y_test
X_train, y_train, X_test, y_test = standardize(X_train, y_train, X_test, y_test)

In [27]:
def data_cleaning_checks(X_train, y_train, X_test, y_test):
    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)
    print("X_test :", X_test.shape)
    print("y_test :", y_test.shape)
    print("NaNs in X:", np.isnan(X_train).sum(), np.isnan(X_test).sum())
    print("Infs in X:", np.isinf(X_train).sum(), np.isinf(X_test).sum())
    print("Min:", X_train.min(), "Max:", X_train.max())
    unique, counts = np.unique(y_train, return_counts=True)
    print("Train balance:", dict(zip(unique, counts)))
    unique, counts = np.unique(y_test, return_counts=True)
    print("Test balance:", dict(zip(unique, counts)))
    i = np.random.randint(len(X_train))
    print("Label:", y_train[i])
    print("Window slice:", X_train[i, 0:5, :5])
    idx = np.random.randint(len(X_train))
    window = X_seq[idx]       # before split
    labels = y_seq[idx]       # before split
    print("Manual majority label:", int(labels.mean() >= 0.5))
    print("Saved label:", y_seq[idx])
    for i in range(5):
        idx = np.random.randint(len(X_train))
        print(idx, y_train[idx], X_train[idx].shape)

In [26]:

# Save everything in one file
np.savez_compressed("Preprocessed_Data/OTIDS_clean_data.npz",
         X_train=X_train, X_test=X_test,
         y_train=y_train, y_test=y_test)

In [30]:
# b_size = 32
# callbacks = [
#     ModelCheckpoint("saved_models/best_model_cnn.keras", monitor='val_accuracy', save_best_only=True, verbose=1),
#     ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
#     EarlyStopping(monitor='val_accuracy', patience=1, verbose=1, restore_best_weights=True)
# ]
# model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
# model.fit(X_train, y_train_cat, batch_size = b_size, epochs = 3, validation_split=0.1, callbacks = callbacks, verbose = 1)

In [31]:
# testing_acc = model.evaluate(X_test,y_test, verbose=1)
# print(f"Test loss: {testing_acc[0]}")
# print(f"Test accuracy: {testing_acc[1]}")

In [32]:
# #examine classification predictions
# y_pred_probs = model.predict(X_test)
# y_pred = np.argmax(y_pred_probs, axis=1)       
# y_true = np.argmax(y_test_cat, axis=1) 
# print(classification_report(y_true, y_pred, target_names=["Normal", "DoS", "Fuzzy", "Impersonation"]))

# #ROC AUC 
# #sensitivity (recall) and 
# # validity (precision)
# from sklearn.metrics import roc_auc_score
# roc_auc = roc_auc_score(y_test_cat, y_pred_probs, multi_class='ovr')
# print(f"ROC AUC Score: {roc_auc}")

In [33]:
from sklearn.metrics import confusion_matrix


cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Normal", "DoS", "Fuzzy", "Impersonation"],
            yticklabels=["Normal", "DoS", "Fuzzy", "Impersonation"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()  

NameError: name 'y_true' is not defined